# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors: Exploration with `mlcroissant`

This notebook demonstrates how to load and explore the FAIR^2 dataset using the `mlcroissant` library. The dataset contains clinical, pathological, and molecular characteristics of 77 cancer survivors with second primary colorectal cancer. We will:

- Access the data using its Croissant schema
- Review available record sets and fields (referenced by their `@id`)
- Extract tabular records into pandas DataFrames
- Run exploratory analysis and visualizations based on the schema

### Dataset Source
The dataset is published with a Croissant schema and can be accessed at:  
https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json

In [ ]:
# Install mlcroissant if not already available
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Croissant schema URL for the dataset
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load the dataset package and metadata
dataset = mlc.Dataset(croissant_url)

# Print dataset name and description
print("Dataset:", dataset.metadata.name)
print("Description:", dataset.metadata.description)


## 2. Data Overview
Review available record sets, their fields, and their `@id`s.

We use the dataset metadata for structured exploration, referencing each entity by its `@id` as per the FAIR^2 schema. (If the schema contains multiple record sets, they are all listed; otherwise, the single main record set is highlighted.)

In [ ]:
# List all record sets in the dataset, showing each one's @id and available fields and columns
record_set_ids = [rs['@id'] for rs in getattr(dataset.metadata, 'recordSet', [])]

if len(record_set_ids) == 0:
    print("No record sets defined directly in the top-level metadata. Attempting to infer from resources...")
    # It's possible the schema exposes data at the distribution/resource level, or the main dataset is a single record set
    # Let's use mlcroissant's listing abilities
    for rs in dataset.list_record_sets():
        print(f"RecordSet @id: {rs['@id']}")
        print("  Name:", rs.get('name', None))
        print("  Fields:")
        for field in rs.get('field', []):
            if isinstance(field, dict):
                print(f"    Field @id: {field.get('@id')} (name: {field.get('name', '')})")
            else:
                print(f"    Field @id: {field}")
        print("  Columns:")
        for col in rs.get('column', []):
            print(f"    Column @id: {col.get('@id')} (name: {col.get('name', '')})")
else:
    print("Defined record sets:", record_set_ids)
    for rs_id in record_set_ids:
        record_set_obj = dataset.get_record_set(rs_id)
        print(f"RecordSet @id: {rs_id}")
        print("  Name:", getattr(record_set_obj, 'name', None))
        print("  Fields (by @id):")
        for field in getattr(record_set_obj, 'field', []):
            if isinstance(field, dict):
                print(f"    {field.get('@id')} (name: {field.get('name', None)})")
            else:
                print(f"    {field}")
        print("  Columns (by @id):")
        for col in getattr(record_set_obj, 'column', []):
            print(f"    {col.get('@id')} (name: {col.get('name', None)})")

# Show an example record using one available record set
print("\nSample records for the first record set (if any):")
record_sets_listed = list(dataset.list_record_sets())
if len(record_sets_listed) > 0:
    rs_id = record_sets_listed[0]['@id']
    for i, rec in enumerate(dataset.records(record_set=rs_id)):
        print(rec)
        if i >= 2:
            break


## 3. Data Extraction

Load data from each record set into a pandas DataFrame. Use the record set and field `@id`s from the overview. All dataframes are keyed by the record set `@id`.

In [ ]:
# List all available record set @id's via mlcroissant helper
record_sets = []
for rs in dataset.list_record_sets():
    record_sets.append(rs['@id'])
print(f"Record set @ids in the dataset: {record_sets}")

dataframes = {}
for record_set_id in record_sets:
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)

# List columns of the first record set
main_rs_id = record_sets[0]
print(f"Columns in DataFrame for record set {main_rs_id}:")
print(dataframes[main_rs_id].columns.tolist())
dataframes[main_rs_id].head()

## 4. Exploratory Data Analysis (EDA)

Apply some useful exploratory steps by referencing fields strictly by their `@id`. We'll filter, normalize, and group records using a numeric and a grouping field (all accesses via DataFrame keys that equal the field `@id`). Adjust variables as needed for your own data exploration.

In [ ]:
# Identify numeric and group fields by their @id
df = dataframes[main_rs_id]
print("Available columns (use @id):", df.columns.tolist())

# For demonstration, attempt to pick numeric and groupable fields
# Let's try to select typical clinical field ids; update if you have field descriptions
possible_numeric_fields = [c for c in df.columns if 'age' in c.lower() or 'interval' in c.lower() or 'years' in c.lower()]
numeric_field = possible_numeric_fields[0] if possible_numeric_fields else df.select_dtypes('number').columns[0]
print(f"Using numeric field @id: {numeric_field}")

# Set threshold for filtering
threshold = 60  # For example, if numeric_field is age
filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold}:")
print(filtered_df.head())

# Normalize selected numeric field
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"Normalized '{numeric_field}' for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Identify a field to group by (categorical)
possible_group_fields = [c for c in df.columns if 'sex' in c.lower() or 'gender' in c.lower() or 'site' in c.lower() or 'location' in c.lower() or 'msi' in c.lower()]
group_field = possible_group_fields[0] if possible_group_fields else df.columns[1]  # pick second column if no good match
print(f"Grouping by field @id: {group_field}")
if group_field in df.columns:
    grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
    print(f"Mean statistics grouped by {group_field}:")
    print(grouped_df.head())

## 5. Visualization

Visualize the distribution of the numeric field and its relation to the chosen group/categorical field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of numeric field
plt.figure(figsize=(8, 4))
sns.histplot(df[numeric_field], bins=10, kde=True)
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.ylabel("Count")
plt.show()

# Boxplot of numeric field by group
if group_field in df.columns:
    plt.figure(figsize=(8, 4))
    sns.boxplot(x=df[group_field], y=df[numeric_field])
    plt.title(f"{numeric_field} by {group_field}")
    plt.xlabel(group_field)
    plt.ylabel(numeric_field)
    plt.show()

## 6. Conclusion

In this notebook, we have:
- Loaded the FAIR^2 dataset directly from its Croissant schema using `mlcroissant`
- Explored its structure by referencing record sets and fields via their `@id` attributes
- Extracted tabular clinical records for analysis
- Demonstrated exploratory data analysis, normalization, grouping, and visualization by referencing all fields by their `@id`s

**Next steps:**
- Expand EDA for specific clinical questions and hypotheses
- Train statistical or machine learning models using the well-defined and FAIR-structured data
- Automate reporting pipelines, using stable `@id`-based references, for reproducible FAIR clinical research workflows.